# A4: Generative Models

In this lab, we will implement and compare **three generative model families** that define the history of AI image generation:

1. **GAN** (Generative Adversarial Network) — adversarial training between a generator and discriminator
2. **DCGAN** (Deep Convolutional GAN) — GANs with convolutional layers for realistic image generation
3. **DDPM** (Denoising Diffusion Probabilistic Model) — the backbone behind Stable Diffusion and DALL-E

---

## Background: The Quest to Generate Images

### Why is Image Generation Hard?

To generate a realistic image, a model must learn the full probability distribution of natural images — an astronomically high-dimensional space. A 32×32 RGB image has **3,072 pixel values**, each ranging 0–255. Yet somehow, a neural network must learn to sample from the tiny subset that looks like real photographs.

Three major approaches have dominated:

| Approach | Core Idea | Pros | Cons |
|---|---|---|---|
| **GAN** | Train generator vs discriminator | Fast, sharp images | Unstable, mode collapse |
| **VAE** | Encode to latent space + decode | Stable, smooth latent | Blurry outputs |
| **Diffusion** | Iteratively denoise from noise | High quality, diverse | Slow sampling |

![Overview of Generative Models](img/generative_overview.png)

---

### The GAN Revolution (2014)

Ian Goodfellow introduced GANs in 2014 with a simple but powerful idea: instead of defining a loss function explicitly, let a **discriminator network** learn the loss. The generator and discriminator play a minimax game — the generator tries to fool the discriminator, and the discriminator tries to catch fakes.

This was revolutionary because it bypassed the intractability of computing image likelihoods. The result: sharp, realistic images that previous methods could not produce.

### From FC-GAN to DCGAN (2015)

The original GAN used fully-connected layers — which worked on MNIST but produced blurry results on complex images. Radford et al. (2015) solved this with **DCGAN**: replace all fully-connected layers with convolutions, add Batch Normalization, and use strided convolutions instead of pooling. These architectural choices made GAN training dramatically more stable.

### The Diffusion Era (2020–present)

Ho et al. (2020) revisited an old idea from thermodynamics: if you gradually add noise to an image until it becomes pure Gaussian noise (forward process), can a neural network learn to reverse this process step by step? The answer was yes — and the result was **DDPM**, which produced images surpassing GANs in quality and diversity, without the training instability. Diffusion is now the backbone of Stable Diffusion, DALL-E 2/3, and Midjourney.

## 📚 Papers & Code References

| Model | Paper | Venue | Code Base |
|---|---|---|---|
| **GAN** | Goodfellow et al. (2014). *Generative Adversarial Nets* | NeurIPS 2014 | From scratch |
| **DCGAN** | Radford et al. (2015). *Unsupervised Representation Learning with DCGANs* | ICLR 2016 | From scratch |
| **VAE** (background) | Kingma & Welling (2013). *Auto-Encoding Variational Bayes* | ICLR 2014 | Conceptual only |
| **DDPM** | Ho et al. (2020). *Denoising Diffusion Probabilistic Models* | NeurIPS 2020 | From scratch |
| **Improved DDPM** (Ex 2) | Nichol & Dhariwal (2021). *Improved DDPMs* | ICML 2021 | Cosine schedule only |

**External code used:**
- GAN training loop structure adapted from: https://github.com/diegoalejogm/gans (MIT)
- DDPM U-Net design inspired by: https://github.com/lucidrains/denoising-diffusion-pytorch (MIT)

**Paper links:**
- GAN: https://arxiv.org/abs/1406.2661
- DCGAN: https://arxiv.org/abs/1511.06434
- DDPM: https://arxiv.org/abs/2006.11239
- Improved DDPM: https://arxiv.org/abs/2102.09672

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os, random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)
os.makedirs('saved', exist_ok=True)

---
## Part 1: GAN — Generative Adversarial Networks

### The Minimax Game

Goodfellow et al. (2014) introduced GANs as a **two-player game**:
- **Generator G**: Takes random noise `z ~ N(0,I)` and produces fake images realistic enough to fool D
- **Discriminator D**: Tries to correctly classify real vs. fake images

They play a minimax game:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

![GAN Architecture Diagram](img/gan_diagram.png)

**Intuition:**
- D wants to output 1 for real images and 0 for fakes → maximize both log terms
- G wants D to output 1 for its fakes → minimize log(1 − D(G(z)))

**Why balance matters:** If D is far superior to G, the gradient signal to G vanishes — G cannot improve because D always outputs ~0 for every fake. They must be roughly balanced throughout training.

![GAN Training Overview](img/gan.png)

**Mode Collapse:** A common failure mode — G learns to produce only a few types of outputs (e.g., only the digit "1"), ignoring the rest of the data distribution.

### Vanilla GAN on MNIST

Let's set up a simple fully-connected GAN to generate handwritten digits.

In [ ]:
# --- Data ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])  # scale to [-1, 1]
])
mnist = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_loader = DataLoader(mnist, batch_size=128, shuffle=True, num_workers=2)

In [ ]:
class Generator(nn.Module):
    """Fully-connected generator: noise z → fake 28x28 image."""
    def __init__(self, z_dim=100, img_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512),  nn.LeakyReLU(0.2),
            nn.Linear(512, 1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim), nn.Tanh()  # output in [-1, 1]
        )
    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """Fully-connected discriminator: image → real/fake probability."""
    def __init__(self, img_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 1024), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(1024, 512),     nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256),      nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),        nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
# --- GAN Setup ---
Z_DIM = 100
G = Generator(Z_DIM).to(device)
D = Discriminator().to(device)

opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

# Fixed noise to track training progress
fixed_noise = torch.randn(64, Z_DIM).to(device)

def show_generated(G, noise, epoch):
    G.eval()
    with torch.no_grad():
        fake = G(noise).view(-1, 1, 28, 28).cpu()
    grid = torchvision.utils.make_grid(fake, nrow=8, normalize=True)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(f'Epoch {epoch} — Generated MNIST')
    plt.axis('off')
    plt.show()
    G.train()

In [ ]:
# --- Training ---
GAN_EPOCHS = 20
g_losses, d_losses = [], []

for epoch in range(GAN_EPOCHS):
    g_ep, d_ep = [], []
    for real_imgs, _ in tqdm(mnist_loader, desc=f'GAN Epoch {epoch+1}/{GAN_EPOCHS}'):
        B = real_imgs.size(0)
        real_imgs = real_imgs.view(B, -1).to(device)
        real_labels = torch.ones(B, 1).to(device)
        fake_labels = torch.zeros(B, 1).to(device)

        # --- Train Discriminator ---
        z = torch.randn(B, Z_DIM).to(device)
        fake_imgs = G(z).detach()  # detach: don't update G here
        d_loss = criterion(D(real_imgs), real_labels) + criterion(D(fake_imgs), fake_labels)
        opt_D.zero_grad(); d_loss.backward(); opt_D.step()

        # --- Train Generator ---
        z = torch.randn(B, Z_DIM).to(device)
        g_loss = criterion(D(G(z)), real_labels)  # G wants D to say "real"
        opt_G.zero_grad(); g_loss.backward(); opt_G.step()

        g_ep.append(g_loss.item()); d_ep.append(d_loss.item())

    g_losses.append(np.mean(g_ep))
    d_losses.append(np.mean(d_ep))
    if (epoch + 1) % 5 == 0:
        show_generated(G, fixed_noise, epoch + 1)

# Plot losses
plt.figure(figsize=(10, 4))
plt.plot(g_losses, label='Generator Loss')
plt.plot(d_losses, label='Discriminator Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Vanilla GAN Training Losses')
plt.legend(); plt.grid(True); plt.show()

## Part 2: DCGAN on CIFAR-10

Fully-connected GANs don't scale to complex RGB images. **DCGAN** (Deep Convolutional GAN, Radford et al. 2015) replaces all linear layers with convolutions:
- **Generator**: Transposed convolutions to upsample from noise → image (100 → 4×4 → 8×8 → 16×16 → 32×32)
- **Discriminator**: Strided convolutions to downsample image → probability (32×32 → 16×16 → 8×8 → 4×4 → 1)

![DCGAN Generator Architecture](img/dcgan_generator.png)

**Key design choices from the DCGAN paper:**
- Batch Normalization in G (except output) and D (except input layer) — stabilizes training
- LeakyReLU (α=0.2) in D, ReLU in G — prevents gradient vanishing
- Tanh output for G, Sigmoid for D — matches [−1, 1] image normalization
- No pooling layers — use strided convolutions instead (let the network learn downsampling)
- No fully connected layers after convolutions

These tricks make the difference between GAN training that explodes and one that converges.

In [ ]:
# --- CIFAR-10 Data ---
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])
cifar = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=cifar_transform)
cifar_loader = DataLoader(cifar, batch_size=128, shuffle=True, num_workers=2)


class DCGenerator(nn.Module):
    """DCGAN Generator: z (100) → 3x32x32 image via transposed convolutions."""
    def __init__(self, z_dim=100, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            # z: (B, 100, 1, 1)
            nn.ConvTranspose2d(z_dim, ngf*4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),        # → 4x4
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),        # → 8x8
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),          # → 16x16
            nn.ConvTranspose2d(ngf, 3, 4, 2, 1, bias=False),
            nn.Tanh()                                     # → 3x32x32
        )
    def forward(self, z): return self.net(z.view(-1, 100, 1, 1))


class DCDiscriminator(nn.Module):
    """DCGAN Discriminator: 3x32x32 → real/fake probability."""
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),             # → 16x16
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),  # → 8x8
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),  # → 4x4
            nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()                                  # → 1x1x1
        )
    def forward(self, x): return self.net(x).view(-1, 1)

In [ ]:
DC_G = DCGenerator().to(device)
DC_D = DCDiscriminator().to(device)
opt_DCG = torch.optim.Adam(DC_G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_DCD = torch.optim.Adam(DC_D.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_noise_dc = torch.randn(64, 100).to(device)

DCGAN_EPOCHS = 20
for epoch in range(DCGAN_EPOCHS):
    for real_imgs, _ in tqdm(cifar_loader, desc=f'DCGAN Epoch {epoch+1}/{DCGAN_EPOCHS}'):
        B = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        real_labels = torch.ones(B, 1).to(device)
        fake_labels = torch.zeros(B, 1).to(device)

        z = torch.randn(B, 100).to(device)
        fake_imgs = DC_G(z).detach()
        d_loss = criterion(DC_D(real_imgs), real_labels) + criterion(DC_D(fake_imgs), fake_labels)
        opt_DCD.zero_grad(); d_loss.backward(); opt_DCD.step()

        z = torch.randn(B, 100).to(device)
        g_loss = criterion(DC_D(DC_G(z)), real_labels)
        opt_DCG.zero_grad(); g_loss.backward(); opt_DCG.step()

    if (epoch + 1) % 5 == 0:
        DC_G.eval()
        with torch.no_grad():
            fake = DC_G(fixed_noise_dc).cpu()
        grid = torchvision.utils.make_grid(fake, nrow=8, normalize=True)
        plt.figure(figsize=(8, 8))
        plt.imshow(grid.permute(1, 2, 0))
        plt.title(f'DCGAN — Epoch {epoch+1}')
        plt.axis('off'); plt.show()
        DC_G.train()

torch.save(DC_G.state_dict(), 'saved/dcgan_cifar10.pt')

---
## Part 3: DDPM — Denoising Diffusion Probabilistic Models

### Why Diffusion?

GANs produce sharp images but suffer from:
1. **Training instability** — the minimax game can oscillate or collapse
2. **Mode collapse** — G ignores parts of the data distribution

Ho et al. (NeurIPS 2020) asked: *"What if we model image generation as iterative denoising?"*

### The Two Processes

![DDPM Forward and Reverse Process](img/ddpm_process.png)

**Forward Process (Fixed, not learned):** Gradually add Gaussian noise over T=1000 steps until the image becomes pure noise.

$$q(x_t \mid x_{t-1}) = \mathcal{N}(x_t;\; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t \mathbf{I})$$

**Shortcut (reparameterization):** Jump directly to any timestep `t`:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\bar{\alpha}_t = \prod_{s=1}^{t}(1-\beta_s)$ is the cumulative signal retention.

**Reverse Process (Learned):** A U-Net $\epsilon_\theta$ predicts the noise added at step `t` and subtracts it to recover `x_{t-1}`.

**Training objective (surprisingly simple!):**

$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}\left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

Just predict the noise. No adversarial game, no mode collapse.

![DDPM Training and Sampling Algorithms](img/ddpm_training.png)

### The Noise Schedule

The β schedule controls how fast we destroy the image. A **linear schedule** increases β from 1e-4 to 0.02. The **cosine schedule** (Nichol & Dhariwal, 2021) decays more gradually at the start, preserving more signal early on.

![Noise Schedule Comparison](img/noise_schedule.png)

In [ ]:
# ====================================================
#  DDPM: Noise Schedule
# ====================================================

T = 1000  # total diffusion timesteps

def linear_beta_schedule(timesteps, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, timesteps)

betas   = linear_beta_schedule(T).to(device)
alphas  = 1.0 - betas
alpha_bar = torch.cumprod(alphas, dim=0)  # ᾱ_t = ∏ α_1 ... α_t
sqrt_alpha_bar     = torch.sqrt(alpha_bar)
sqrt_one_minus_ab  = torch.sqrt(1.0 - alpha_bar)

# Visualize the forward process
mnist_sample, _ = next(iter(DataLoader(mnist, batch_size=1, shuffle=True)))
img = mnist_sample[0].to(device)

fig, axes = plt.subplots(1, 7, figsize=(14, 2))
timesteps_to_show = [0, 100, 200, 400, 600, 800, 999]
for ax, t in zip(axes, timesteps_to_show):
    eps = torch.randn_like(img)
    noisy = sqrt_alpha_bar[t] * img + sqrt_one_minus_ab[t] * eps
    ax.imshow(noisy.squeeze().cpu().detach().numpy(), cmap='gray')
    ax.set_title(f't={t}')
    ax.axis('off')
plt.suptitle('Forward Process: Gradually adding noise', y=1.02)
plt.tight_layout(); plt.show()

## DDPM Step 2: The U-Net

The denoising network is a **U-Net** — an encoder-decoder architecture with skip connections. It takes two inputs:
- The noisy image `x_t`
- The timestep `t` (encoded as a sinusoidal embedding, like positional encoding in Transformers)

This lets the network know "how noisy" the image is and predict the appropriate amount of noise to remove.

![U-Net Architecture](img/unet_arch.jpg)

![DDPM Algorithm](img/ddpm_algo.png)

We build a simplified U-Net for MNIST (28×28, grayscale) to keep training tractable. In practice, DDPM uses a full-scale U-Net with attention layers at multiple resolutions.

In [ ]:
class SinusoidalEmbedding(nn.Module):
    """Encodes scalar timestep t into a fixed-size vector using sinusoidal frequencies."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -torch.arange(half, device=t.device).float() * (torch.log(torch.tensor(10000.0)) / (half - 1))
        )
        args = t.float()[:, None] * freqs[None]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResBlock(nn.Module):
    """Residual block with timestep conditioning."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.residual = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.norm1 = nn.GroupNorm(8, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)

    def forward(self, x, t_emb):
        h = self.norm1(F.silu(self.conv1(x)))
        h = h + self.time_mlp(t_emb)[:, :, None, None]  # add time embedding
        h = self.norm2(F.silu(self.conv2(h)))
        return h + self.residual(x)


class SimpleUNet(nn.Module):
    """Lightweight U-Net for DDPM on MNIST."""
    def __init__(self, in_ch=1, base_ch=64, time_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )
        # Encoder
        self.enc1 = ResBlock(in_ch, base_ch, time_dim)
        self.enc2 = ResBlock(base_ch, base_ch*2, time_dim)
        self.down  = nn.MaxPool2d(2)
        # Bottleneck
        self.bot   = ResBlock(base_ch*2, base_ch*4, time_dim)
        # Decoder
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2  = ResBlock(base_ch*4 + base_ch*2, base_ch*2, time_dim)
        self.dec1  = ResBlock(base_ch*2 + base_ch, base_ch, time_dim)
        self.out   = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_embed(t)
        # Encode
        e1 = self.enc1(x, t_emb)          # (B, 64, 28, 28)
        e2 = self.enc2(self.down(e1), t_emb)  # (B, 128, 14, 14)
        # Bottleneck
        b  = self.bot(self.down(e2), t_emb)   # (B, 256, 7, 7)
        # Decode with skip connections
        d2 = self.dec2(torch.cat([self.up(b), e2], dim=1), t_emb)   # (B, 128, 14, 14)
        d1 = self.dec1(torch.cat([self.up(d2), e1], dim=1), t_emb)  # (B, 64, 28, 28)
        return self.out(d1)  # predict noise ε, same shape as input

## DDPM Step 3: Training

The training loop is elegant:
1. Sample a clean image `x_0` and a random timestep `t`
2. Add noise to get `x_t` using the shortcut formula
3. Ask the U-Net to predict the noise `ε` that was added
4. Loss = MSE(predicted noise, actual noise)

That's all! No adversarial game, no posterior collapse — just simple regression.

In [ ]:
unet = SimpleUNet().to(device)
opt_ddpm = torch.optim.Adam(unet.parameters(), lr=2e-4)
print(f'U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}')


def q_sample(x0, t, noise=None):
    """Forward process: add noise to x0 at timestep t."""
    if noise is None:
        noise = torch.randn_like(x0)
    return sqrt_alpha_bar[t][:, None, None, None] * x0 + \
           sqrt_one_minus_ab[t][:, None, None, None] * noise


DDPM_EPOCHS = 10  # increase to 50+ for good results
ddpm_losses = []

for epoch in range(DDPM_EPOCHS):
    unet.train()
    ep_loss = []
    for x0, _ in tqdm(mnist_loader, desc=f'DDPM Epoch {epoch+1}/{DDPM_EPOCHS}'):
        x0 = x0.to(device)
        B  = x0.size(0)

        # Sample random timesteps for each image in batch
        t = torch.randint(0, T, (B,), device=device)

        # Add noise (forward process)
        noise  = torch.randn_like(x0)
        x_t    = q_sample(x0, t, noise)

        # Predict the noise
        pred_noise = unet(x_t, t)

        # Simple MSE loss on noise prediction
        loss = F.mse_loss(pred_noise, noise)

        opt_ddpm.zero_grad()
        loss.backward()
        opt_ddpm.step()
        ep_loss.append(loss.item())

    ddpm_losses.append(np.mean(ep_loss))
    print(f'Epoch {epoch+1:03d} | Loss: {np.mean(ep_loss):.4f}')

torch.save(unet.state_dict(), 'saved/ddpm_mnist.pt')

plt.figure(figsize=(8, 4))
plt.plot(ddpm_losses, marker='o', color='orange')
plt.title('DDPM Training Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.grid(True); plt.show()

## DDPM Step 4: Sampling (Generation)

To generate a new image:
1. Start with pure Gaussian noise `x_T`
2. Apply the U-Net 1000 times in reverse, each time subtracting a predicted noise
3. The image gradually becomes cleaner

The reverse step formula:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z, \quad z \sim \mathcal{N}(0, I)$$

![DDPM Sampling Process](img/ddpm_sampling.png)

In [ ]:
unet.load_state_dict(torch.load('saved/ddpm_mnist.pt', map_location=device))
unet.eval()

# Precompute values for reverse process
sqrt_recip_alphas     = torch.sqrt(1.0 / alphas)
posterior_variance    = betas * (1.0 - F.pad(alpha_bar[:-1], (1, 0), value=1.0)) / (1.0 - alpha_bar)


@torch.no_grad()
def p_sample(x_t, t_scalar):
    """One reverse diffusion step: x_t → x_{t-1}."""
    t_batch = torch.full((x_t.size(0),), t_scalar, device=device, dtype=torch.long)
    pred_noise = unet(x_t, t_batch)

    # Compute mean of reverse process
    coeff = betas[t_scalar] / sqrt_one_minus_ab[t_scalar]
    mean  = sqrt_recip_alphas[t_scalar] * (x_t - coeff * pred_noise)

    if t_scalar == 0:
        return mean
    noise = torch.randn_like(x_t)
    return mean + torch.sqrt(posterior_variance[t_scalar]) * noise


@torch.no_grad()
def generate(n_samples=64):
    """Full generation: start from noise, run T reverse steps."""
    x = torch.randn(n_samples, 1, 28, 28).to(device)
    for t in tqdm(reversed(range(T)), total=T, desc='Sampling'):
        x = p_sample(x, t)
    return x


samples = generate(64)
grid = torchvision.utils.make_grid(samples, nrow=8, normalize=True)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0).cpu())
plt.title('DDPM Generated MNIST Samples')
plt.axis('off'); plt.show()

## Visualize the Denoising Process

Let's watch the reverse process in action — from pure noise to a digit.

In [ ]:
@torch.no_grad()
def generate_with_trajectory(n_samples=8):
    """Generate images and record snapshots along the way."""
    x = torch.randn(n_samples, 1, 28, 28).to(device)
    snapshots = []
    show_at = {999, 800, 600, 400, 200, 100, 50, 0}
    for t in reversed(range(T)):
        x = p_sample(x, t)
        if t in show_at:
            snapshots.append((t, x.cpu().clone()))
    return snapshots


snapshots = generate_with_trajectory(8)
fig, axes = plt.subplots(8, len(snapshots), figsize=(len(snapshots)*1.5, 12))

for col, (t, imgs) in enumerate(snapshots):
    for row in range(8):
        ax = axes[row][col]
        ax.imshow(imgs[row].squeeze().numpy(), cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(f't={t}', fontsize=9)

plt.suptitle('Reverse Diffusion Process: Noise → Digit', fontsize=13)
plt.tight_layout(); plt.show()

# Exercises

## Exercise 1: GAN Mode Collapse

Mode collapse is when the generator only produces a few types of outputs (e.g., only generates the digit "1" regardless of noise input).

a) After training the Vanilla GAN, generate 1000 images and classify them using a pretrained MNIST classifier. Plot a histogram of the predicted class distribution. Does your GAN cover all 10 digits evenly?

b) Intentionally cause mode collapse by making the discriminator 3× stronger (triple the learning rate of D). What do you observe?

c) Describe two techniques that help prevent mode collapse (from your lecture notes).

---

## Exercise 2: DDPM Noise Schedule Ablation

The noise schedule determines how quickly the image is destroyed.

a) Implement and compare two schedules:
- **Linear**: β increases linearly from 1e-4 to 0.02 (already implemented)
- **Cosine**: ᾱ_t = cos²(((t/T + 0.008) / 1.008) × π/2) — proposed by Nichol & Dhariwal (2021) to prevent too-rapid early noise

```python
def cosine_beta_schedule(timesteps, s=0.008):
    t = torch.linspace(0, timesteps, timesteps + 1)
    alphas_bar = torch.cos(((t / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_bar = alphas_bar / alphas_bar[0]
    betas = 1 - (alphas_bar[1:] / alphas_bar[:-1])
    return torch.clamp(betas, 0.0001, 0.9999)
```

b) Plot both ᾱ_t curves side by side. Describe the key difference.

c) Train with both schedules and compare the quality of generated samples. Which looks better? Why?

---

## Exercise 3: DDPM on CIFAR-10

Our DDPM was trained on MNIST (grayscale 28×28). Now scale it up to CIFAR-10 (RGB 32×32).

a) Modify `SimpleUNet` to handle 3-channel (RGB) input by changing `in_ch=3`.

b) Train for at least 20 epochs on CIFAR-10. Show a grid of 64 generated samples.

c) Compare training time per epoch between MNIST and CIFAR-10. What is the bottleneck?

---

## Exercise 4 (Challenge): GAN vs Diffusion — Side-by-Side Comparison

a) Measure and compare:

| Metric | Vanilla GAN | DCGAN | DDPM |
|---|---|---|---|
| Training time (20 epochs) | ? | ? | ? |
| Inference time (64 samples) | ? | ? | ? |
| Visual quality (subjective 1–5) | ? | ? | ? |
| Mode diversity (# distinct digits) | ? | ? | ? |

b) Based on your experiments, when would you choose GAN over Diffusion? Give a real-world use case where speed matters more than quality.

c) The lecture mentioned VAEs are used as the compression engine inside **Latent Diffusion Models** (Stable Diffusion). Read the Stable Diffusion paper abstract (Rombach et al., 2022) and explain in 3 sentences why running diffusion in latent space is more efficient than pixel space.

---

## Submission

Submit your work to GitHub. Your repository should contain:

### 1. Training Script (`train.py`)

```bash
# Train Vanilla GAN on MNIST
python3 train.py --model gan      --dataset mnist   --epochs 20 --train

# Train DCGAN on CIFAR-10
python3 train.py --model dcgan    --dataset cifar10 --epochs 20 --train

# Train DDPM on MNIST
python3 train.py --model ddpm     --dataset mnist   --epochs 20 --train

# Train DDPM on CIFAR-10 (Exercise 3)
python3 train.py --model ddpm     --dataset cifar10 --epochs 20 --train

# DDPM with cosine schedule (Exercise 2)
python3 train.py --model ddpm     --dataset mnist   --epochs 20 --schedule cosine --train

# Generate samples
python3 train.py --model ddpm     --dataset mnist   --weights saved/ddpm_mnist.pt --generate --n 64
```

### 2. `README.md`

Your `README.md` must include:

**Commands used** (exact commands you ran)

**Results table:**

| Model | Dataset | FID / Visual Quality | Training Time | Notes |
|---|---|---|---|---|
| Vanilla GAN | MNIST | ? | ? | mode collapse check |
| DCGAN | CIFAR-10 | ? | ? | conv-based |
| DDPM (linear) | MNIST | ? | ? | baseline |
| DDPM (cosine) | MNIST | ? | ? | schedule ablation |
| DDPM | CIFAR-10 | ? | ? | scaled up |

**Visualizations** (include in README or as separate image files):
- Generated image grids for all three models
- Noise schedule comparison plot (ᾱ_t linear vs cosine)
- Mode collapse histogram (Exercise 1a)
- Denoising trajectory (from pure noise → final image)

**Discussion** (3–5 sentences): Based on your experiments, which generative model would you use for a real-world image synthesis application, and why?